# Environmental Impact — Batching Dispatch Optimization
Quantifies CO2e emissions saved from batching-optimized dispatch vs greedy baseline.
Scope: Tank-to-wheel (Scope 1) combustion emissions only. Excludes upstream
extraction/refining (well-to-tank). NOx/CO/PM2.5 excluded — BS6 norms are
certification limits, not operational factors. Noted as future work.

**Structure**
- **Section 1 — Constants**
- **Section 2 — HD City: Per-Scenario Emissions**
- **Section 3 — HYD City: Per-Scenario Emissions**
- **Section 4 — National Weighted Average**
- **Section 5 — PRIMARY: Avg(P, Q) National Projection**
- **Section 6 — P–Q Range + Sub-Models**
- **Section 7 — Equivalent Trees + Master Summary**

---
## Section 1 — Constants

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json, os

plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#cccccc",
    "xtick.color":      "#cccccc",
    "ytick.color":      "#cccccc",
    "text.color":       "#ffffff",
    "grid.color":       "#333333",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
})

GREY  = "#555555"
GREEN = "#00c896"
DAYS_PER_YEAR = 365

# ── Emission factor ───────────────────────────────────────────────────────────
# Source: UK Government GHG Conversion Factors for Company Reporting 2026
#         (DESNZ/DEFRA, published 31 July 2026)
#         Sheet: Fuels → Liquid fuels → "Petrol (100% mineral petrol)"
#         Unit: litres, Net CV basis
#         NOTE: "Petrol (average biofuel blend)" NOT used — reflects UK-mandated
#         blending, not representative of Indian petrol composition.
PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE = 2.35372

# ── Fuel efficiency ───────────────────────────────────────────────────────────
# Same constant used in financial model fuel cost derivation — 100% motorcycle
# fleet (simulation_config.json, both cities)
MILEAGE_KM_PER_LITRE = 40.0

# ── Derived emission factor per km ───────────────────────────────────────────
EMISSION_FACTOR_PER_KM = round(PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE /
                               MILEAGE_KM_PER_LITRE, 4)  # kg CO2e/km

# ── Equivalent trees constant ─────────────────────────────────────────────────
# Source: US EPA GHG Equivalencies Calculator (updated Nov 2024)
#         Section: "Number of urban tree seedlings grown for 10 years"
#         Weighted avg: 11% coniferous (23.2 lbs C/tree/yr) +
#                       89% deciduous (38.0 lbs C/tree/yr), US urban tree mix
#         Converted via molecular weight ratio (44/12) and lbs→metric tons
#         Primary source: McPherson, van Doorn & Peper (2016),
#                         Gen. Tech. Rep. PSW-GTR-253, USDA Forest Service
#         NOTE: Illustrative/cosmetic framing only — not analytically load-bearing
TREE_CO2_KG_PER_YEAR = 60.0  # kg CO2e absorbed per urban tree per year

print("── Environmental Impact Constants ───────────────────────────────")
print(f"  Emission factor      : {PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE} kg CO2e/litre")
print(f"  Fuel efficiency      : {MILEAGE_KM_PER_LITRE} km/litre")
print(f"  Emission factor/km   : {EMISSION_FACTOR_PER_KM} kg CO2e/km")
print(f"  Tree absorption      : {TREE_CO2_KG_PER_YEAR} kg CO2e/tree/year")

---
## Section 2 — HD City: Per-Scenario Emissions
Locked KM values from HD_Financial_Model.ipynb (baseline_metrics.json + smart_metrics_p/q.json)

In [ ]:
# ── HD locked KM values ───────────────────────────────────────────────────────
HD_DAILY_ORDERS   = 1_515
HD_GREEDY_KM      = 11_882.28
HD_P_KM           = 11_809.26
HD_Q_KM           = 11_738.01

# ── HD daily emissions (kg CO2e) ──────────────────────────────────────────────
HD_GREEDY_EMISSIONS   = HD_GREEDY_KM * EMISSION_FACTOR_PER_KM
HD_P_EMISSIONS        = HD_P_KM      * EMISSION_FACTOR_PER_KM
HD_Q_EMISSIONS        = HD_Q_KM      * EMISSION_FACTOR_PER_KM
HD_AVG_EMISSIONS      = (HD_P_EMISSIONS + HD_Q_EMISSIONS) / 2

# ── HD daily fuel litres ──────────────────────────────────────────────────────
HD_GREEDY_LITRES      = HD_GREEDY_KM / MILEAGE_KM_PER_LITRE
HD_P_LITRES           = HD_P_KM      / MILEAGE_KM_PER_LITRE
HD_Q_LITRES           = HD_Q_KM      / MILEAGE_KM_PER_LITRE

# ── HD daily delta ────────────────────────────────────────────────────────────
HD_DELTA_EMISSIONS_P  = HD_GREEDY_EMISSIONS - HD_P_EMISSIONS
HD_DELTA_EMISSIONS_Q  = HD_GREEDY_EMISSIONS - HD_Q_EMISSIONS
HD_DELTA_EMISSIONS_AVG= HD_GREEDY_EMISSIONS - HD_AVG_EMISSIONS
HD_DELTA_LITRES_P     = HD_GREEDY_LITRES - HD_P_LITRES
HD_DELTA_LITRES_Q     = HD_GREEDY_LITRES - HD_Q_LITRES

print("══ HD (Hubli-Dharwad) — Daily Emissions ════════════════════════")
print(f"  Greedy  : {HD_GREEDY_EMISSIONS:.2f} kg CO2e/day  |  {HD_GREEDY_LITRES:.2f} L/day")
print(f"  Smart-P : {HD_P_EMISSIONS:.2f} kg CO2e/day  |  {HD_P_LITRES:.2f} L/day")
print(f"  Smart-Q : {HD_Q_EMISSIONS:.2f} kg CO2e/day  |  {HD_Q_LITRES:.2f} L/day")
print(f"  Smart-Avg: {HD_AVG_EMISSIONS:.2f} kg CO2e/day")
print()
print(f"  Δ Emissions saved [P]   : {HD_DELTA_EMISSIONS_P:.2f} kg CO2e/day  ({HD_DELTA_EMISSIONS_P/HD_GREEDY_EMISSIONS*100:.2f}%)")
print(f"  Δ Emissions saved [Q]   : {HD_DELTA_EMISSIONS_Q:.2f} kg CO2e/day  ({HD_DELTA_EMISSIONS_Q/HD_GREEDY_EMISSIONS*100:.2f}%)")
print(f"  Δ Emissions saved [Avg] : {HD_DELTA_EMISSIONS_AVG:.2f} kg CO2e/day")
print(f"  Δ Fuel saved [P–Q]      : {HD_DELTA_LITRES_P:.2f} – {HD_DELTA_LITRES_Q:.2f} L/day")

In [ ]:
# ── Chart: HD emissions comparison ───────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("HD Environmental Impact — Daily Emissions & Fuel (Hubli-Dharwad)",
             fontsize=12, fontweight="bold", color="white", y=1.02)

scenarios  = ["Greedy", "Smart-Q", "Smart-Avg", "Smart-P"]
emissions  = [HD_GREEDY_EMISSIONS, HD_Q_EMISSIONS, HD_AVG_EMISSIONS, HD_P_EMISSIONS]
litres     = [HD_GREEDY_LITRES,    HD_Q_LITRES,    (HD_P_LITRES+HD_Q_LITRES)/2, HD_P_LITRES]
bar_colors = [GREY, "#4488ff", GREEN, "#ff9900"]

# — Panel 1: Daily emissions ──────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(scenarios, emissions, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Daily CO2e Emissions (kg)", fontsize=10, pad=8)
ax.set_ylabel("kg CO2e")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_ylim(min(emissions)*0.97, max(emissions)*1.03)
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, emissions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(emissions)*0.003,
            f"{val:,.1f}", ha="center", va="bottom", fontsize=8, color="white")

# — Panel 2: Daily fuel litres ────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(scenarios, litres, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Daily Fuel Consumed (litres)", fontsize=10, pad=8)
ax.set_ylabel("litres")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.1f} L"))
ax.set_ylim(min(litres)*0.97, max(litres)*1.03)
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, litres):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(litres)*0.003,
            f"{val:,.1f}", ha="center", va="bottom", fontsize=8, color="white")

# — Panel 3: P–Q emission delta ───────────────────────────────────────────────
ax = axes[2]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Daily Emissions Saved [Avg]", fontsize=10, pad=8)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=1.5,
             edgecolor=GREEN, facecolor="#1a1a1a"))
ax.text(0.5, 0.78, f"{HD_DELTA_EMISSIONS_AVG:.2f}",
        ha="center", va="center", fontsize=26, fontweight="bold", color=GREEN)
ax.text(0.5, 0.61, "kg CO2e saved/day",
        ha="center", va="center", fontsize=11, color="#cccccc")
ax.text(0.5, 0.45, f"Range: {HD_DELTA_EMISSIONS_P:.2f} – {HD_DELTA_EMISSIONS_Q:.2f} kg [P–Q]",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.30, f"Fuel saved: {HD_DELTA_LITRES_P:.2f}–{HD_DELTA_LITRES_Q:.2f} L/day",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.15, f"Annual: {HD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000:.3f} tonnes CO2e/yr",
        ha="center", va="center", fontsize=9, color="#aaaaaa")

plt.tight_layout()
plt.savefig("hd_environmental_impact.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → hd_environmental_impact.png")

---
## Section 3 — HYD City: Per-Scenario Emissions
Locked KM values from HYD_Financial_Model.ipynb (baseline_metrics.json + smart_metrics_p/q.json)

In [ ]:
# ── HYD locked KM values ──────────────────────────────────────────────────────
HYD_DAILY_ORDERS  = 16_856
HYD_GREEDY_KM     = 86_196.76
HYD_P_KM          = 85_454.52
HYD_Q_KM          = 85_285.09

# ── HYD daily emissions (kg CO2e) ─────────────────────────────────────────────
HYD_GREEDY_EMISSIONS   = HYD_GREEDY_KM * EMISSION_FACTOR_PER_KM
HYD_P_EMISSIONS        = HYD_P_KM      * EMISSION_FACTOR_PER_KM
HYD_Q_EMISSIONS        = HYD_Q_KM      * EMISSION_FACTOR_PER_KM
HYD_AVG_EMISSIONS      = (HYD_P_EMISSIONS + HYD_Q_EMISSIONS) / 2

# ── HYD daily fuel litres ─────────────────────────────────────────────────────
HYD_GREEDY_LITRES      = HYD_GREEDY_KM / MILEAGE_KM_PER_LITRE
HYD_P_LITRES           = HYD_P_KM      / MILEAGE_KM_PER_LITRE
HYD_Q_LITRES           = HYD_Q_KM      / MILEAGE_KM_PER_LITRE

# ── HYD daily delta ───────────────────────────────────────────────────────────
HYD_DELTA_EMISSIONS_P   = HYD_GREEDY_EMISSIONS - HYD_P_EMISSIONS
HYD_DELTA_EMISSIONS_Q   = HYD_GREEDY_EMISSIONS - HYD_Q_EMISSIONS
HYD_DELTA_EMISSIONS_AVG = HYD_GREEDY_EMISSIONS - HYD_AVG_EMISSIONS
HYD_DELTA_LITRES_P      = HYD_GREEDY_LITRES - HYD_P_LITRES
HYD_DELTA_LITRES_Q      = HYD_GREEDY_LITRES - HYD_Q_LITRES

print("══ HYD (Hyderabad) — Daily Emissions ═══════════════════════════")
print(f"  Greedy  : {HYD_GREEDY_EMISSIONS:.2f} kg CO2e/day  |  {HYD_GREEDY_LITRES:.2f} L/day")
print(f"  Smart-P : {HYD_P_EMISSIONS:.2f} kg CO2e/day  |  {HYD_P_LITRES:.2f} L/day")
print(f"  Smart-Q : {HYD_Q_EMISSIONS:.2f} kg CO2e/day  |  {HYD_Q_LITRES:.2f} L/day")
print(f"  Smart-Avg: {HYD_AVG_EMISSIONS:.2f} kg CO2e/day")
print()
print(f"  Δ Emissions saved [P]   : {HYD_DELTA_EMISSIONS_P:.2f} kg CO2e/day  ({HYD_DELTA_EMISSIONS_P/HYD_GREEDY_EMISSIONS*100:.2f}%)")
print(f"  Δ Emissions saved [Q]   : {HYD_DELTA_EMISSIONS_Q:.2f} kg CO2e/day  ({HYD_DELTA_EMISSIONS_Q/HYD_GREEDY_EMISSIONS*100:.2f}%)")
print(f"  Δ Emissions saved [Avg] : {HYD_DELTA_EMISSIONS_AVG:.2f} kg CO2e/day")
print(f"  Δ Fuel saved [P–Q]      : {HYD_DELTA_LITRES_P:.2f} – {HYD_DELTA_LITRES_Q:.2f} L/day")

In [ ]:
# ── Chart: HYD emissions comparison ──────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("HYD Environmental Impact — Daily Emissions & Fuel (Hyderabad)",
             fontsize=12, fontweight="bold", color="white", y=1.02)

hyd_emissions  = [HYD_GREEDY_EMISSIONS, HYD_Q_EMISSIONS, HYD_AVG_EMISSIONS, HYD_P_EMISSIONS]
hyd_litres     = [HYD_GREEDY_LITRES, HYD_Q_LITRES, (HYD_P_LITRES+HYD_Q_LITRES)/2, HYD_P_LITRES]

# — Panel 1: Daily emissions ──────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(scenarios, hyd_emissions, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Daily CO2e Emissions (kg)", fontsize=10, pad=8)
ax.set_ylabel("kg CO2e")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_ylim(min(hyd_emissions)*0.97, max(hyd_emissions)*1.03)
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, hyd_emissions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(hyd_emissions)*0.003,
            f"{val:,.1f}", ha="center", va="bottom", fontsize=8, color="white")

# — Panel 2: Daily fuel litres ────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(scenarios, hyd_litres, color=bar_colors, width=0.5, zorder=3)
ax.set_title("Daily Fuel Consumed (litres)", fontsize=10, pad=8)
ax.set_ylabel("litres")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.1f} L"))
ax.set_ylim(min(hyd_litres)*0.97, max(hyd_litres)*1.03)
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, hyd_litres):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(hyd_litres)*0.003,
            f"{val:,.1f}", ha="center", va="bottom", fontsize=8, color="white")

# — Panel 3: Emissions delta gauge ────────────────────────────────────────────
ax = axes[2]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Daily Emissions Saved [Avg]", fontsize=10, pad=8)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=1.5,
             edgecolor=GREEN, facecolor="#1a1a1a"))
ax.text(0.5, 0.78, f"{HYD_DELTA_EMISSIONS_AVG:.2f}",
        ha="center", va="center", fontsize=26, fontweight="bold", color=GREEN)
ax.text(0.5, 0.61, "kg CO2e saved/day",
        ha="center", va="center", fontsize=11, color="#cccccc")
ax.text(0.5, 0.45, f"Range: {HYD_DELTA_EMISSIONS_P:.2f} – {HYD_DELTA_EMISSIONS_Q:.2f} kg [P–Q]",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.30, f"Fuel saved: {HYD_DELTA_LITRES_P:.2f}–{HYD_DELTA_LITRES_Q:.2f} L/day",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.15, f"Annual: {HYD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000:.3f} tonnes CO2e/yr",
        ha="center", va="center", fontsize=9, color="#aaaaaa")

plt.tight_layout()
plt.savefig("hyd_environmental_impact.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → hyd_environmental_impact.png")

---
## Section 4 — National Weighted Average
Order-volume weighted (HD: 1,515 / HYD: 16,856) — consistent with national financial model blending.

In [ ]:
# ── National weighting (order-volume based) ───────────────────────────────────
TOTAL_SIM_ORDERS = HD_DAILY_ORDERS + HYD_DAILY_ORDERS
HD_WEIGHT        = HD_DAILY_ORDERS  / TOTAL_SIM_ORDERS
HYD_WEIGHT       = HYD_DAILY_ORDERS / TOTAL_SIM_ORDERS

# National daily orders (3-yr avg FY24-FY26 — Eternal Annual Reports)
NATIONAL_DAILY_ORDERS = ((753e6 + 853e6 + 979e6) / 3) / DAYS_PER_YEAR

# ── Blended per-order emission delta ─────────────────────────────────────────
HD_DELTA_PER_ORDER_P   = HD_DELTA_EMISSIONS_P   / HD_DAILY_ORDERS
HD_DELTA_PER_ORDER_Q   = HD_DELTA_EMISSIONS_Q   / HD_DAILY_ORDERS
HD_DELTA_PER_ORDER_AVG = HD_DELTA_EMISSIONS_AVG / HD_DAILY_ORDERS

HYD_DELTA_PER_ORDER_P   = HYD_DELTA_EMISSIONS_P   / HYD_DAILY_ORDERS
HYD_DELTA_PER_ORDER_Q   = HYD_DELTA_EMISSIONS_Q   / HYD_DAILY_ORDERS
HYD_DELTA_PER_ORDER_AVG = HYD_DELTA_EMISSIONS_AVG / HYD_DAILY_ORDERS

BLENDED_DELTA_PER_ORDER_P   = (HD_DAILY_ORDERS * HD_DELTA_PER_ORDER_P   +
                                HYD_DAILY_ORDERS * HYD_DELTA_PER_ORDER_P)   / TOTAL_SIM_ORDERS
BLENDED_DELTA_PER_ORDER_Q   = (HD_DAILY_ORDERS * HD_DELTA_PER_ORDER_Q   +
                                HYD_DAILY_ORDERS * HYD_DELTA_PER_ORDER_Q)   / TOTAL_SIM_ORDERS
BLENDED_DELTA_PER_ORDER_AVG = (HD_DAILY_ORDERS * HD_DELTA_PER_ORDER_AVG +
                                HYD_DAILY_ORDERS * HYD_DELTA_PER_ORDER_AVG) / TOTAL_SIM_ORDERS

print("── Weighting ────────────────────────────────────────────────────")
print(f"  HD  weight : {HD_WEIGHT*100:.1f}%  ({HD_DAILY_ORDERS:,} orders)")
print(f"  HYD weight : {HYD_WEIGHT*100:.1f}%  ({HYD_DAILY_ORDERS:,} orders)")
print()
print("── Blended per-order emission delta (kg CO2e/order) ─────────────")
print(f"  [Q]   : {BLENDED_DELTA_PER_ORDER_Q:.6f} kg CO2e/order")
print(f"  [Avg] : {BLENDED_DELTA_PER_ORDER_AVG:.6f} kg CO2e/order")
print(f"  [P]   : {BLENDED_DELTA_PER_ORDER_P:.6f} kg CO2e/order")
print()
print(f"  National daily orders (3-yr avg) : {NATIONAL_DAILY_ORDERS:,.0f}")

---
## Section 5 — PRIMARY: Avg(P, Q) National Projection

In [ ]:
# ── National daily + annual emissions saved ───────────────────────────────────

nat_daily_co2e_P   = NATIONAL_DAILY_ORDERS * BLENDED_DELTA_PER_ORDER_P
nat_daily_co2e_Q   = NATIONAL_DAILY_ORDERS * BLENDED_DELTA_PER_ORDER_Q
nat_daily_co2e_avg = NATIONAL_DAILY_ORDERS * BLENDED_DELTA_PER_ORDER_AVG

nat_annual_co2e_P   = nat_daily_co2e_P   * DAYS_PER_YEAR
nat_annual_co2e_Q   = nat_daily_co2e_Q   * DAYS_PER_YEAR
nat_annual_co2e_avg = nat_daily_co2e_avg * DAYS_PER_YEAR

# Fuel litres saved nationally
nat_daily_litres_P   = nat_daily_co2e_P   / PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE
nat_daily_litres_Q   = nat_daily_co2e_Q   / PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE
nat_daily_litres_avg = nat_daily_co2e_avg / PETROL_EMISSION_FACTOR_KG_CO2E_PER_LITRE

nat_annual_litres_avg = nat_daily_litres_avg * DAYS_PER_YEAR

# Trees equivalent
trees_avg = nat_annual_co2e_avg / TREE_CO2_KG_PER_YEAR
trees_P   = nat_annual_co2e_P   / TREE_CO2_KG_PER_YEAR
trees_Q   = nat_annual_co2e_Q   / TREE_CO2_KG_PER_YEAR

print("══ PRIMARY — National Emissions Saved Avg(P, Q) ════════════════")
print(f"  Daily CO2e saved [Avg]   : {nat_daily_co2e_avg:,.2f} kg CO2e/day")
print(f"  Annual CO2e saved [Avg]  : {nat_annual_co2e_avg:,.2f} kg CO2e/year")
print(f"  Annual CO2e saved [Avg]  : {nat_annual_co2e_avg/1000:,.3f} tonnes CO2e/year")
print(f"  Annual fuel saved [Avg]  : {nat_annual_litres_avg:,.0f} litres/year")
print(f"  Equivalent trees [Avg]   : {trees_avg:,.0f} trees")
print()
print(f"  Range [Q–P]: {nat_annual_co2e_Q/1000:,.3f} – {nat_annual_co2e_P/1000:,.3f} tonnes CO2e/year")
print(f"  Trees range : {trees_Q:,.0f} – {trees_P:,.0f} trees")

In [ ]:
# ── Chart: National primary projection ───────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("National Environmental Impact — PRIMARY: Avg(P, Q) Projection",
             fontsize=12, fontweight="bold", color="white", y=1.02)

# — Panel 1: HD vs HYD per-order delta comparison ─────────────────────────────
ax = axes[0]
city_labels = ["HD\n(Tier 2)", "HYD\n(Tier 1)", "Blended\n(National)"]
avg_deltas  = [HD_DELTA_PER_ORDER_AVG, HYD_DELTA_PER_ORDER_AVG, BLENDED_DELTA_PER_ORDER_AVG]
p_deltas    = [HD_DELTA_PER_ORDER_P,   HYD_DELTA_PER_ORDER_P,   BLENDED_DELTA_PER_ORDER_P]
q_deltas    = [HD_DELTA_PER_ORDER_Q,   HYD_DELTA_PER_ORDER_Q,   BLENDED_DELTA_PER_ORDER_Q]
x     = np.arange(len(city_labels))
width = 0.25
ax.bar(x - width, q_deltas,   width, label="Q (Conservative)", color="#4488ff", zorder=3)
ax.bar(x,         avg_deltas, width, label="Avg P+Q (Primary)", color=GREEN,    zorder=3)
ax.bar(x + width, p_deltas,   width, label="P (Aggressive)",   color="#ff9900", zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(city_labels, fontsize=9)
ax.set_title("Per-Order CO2e Saved (kg CO2e/order)", fontsize=10, pad=8)
ax.set_ylabel("kg CO2e per order")
ax.legend(fontsize=8, facecolor="#1a1a1a", edgecolor="#444")
ax.grid(axis="y", zorder=0)
for i, (q, avg, p) in enumerate(zip(q_deltas, avg_deltas, p_deltas)):
    ax.text(i-width, q   + max(p_deltas)*0.02, f"{q:.5f}",   ha="center", fontsize=7, color="#4488ff")
    ax.text(i,       avg + max(p_deltas)*0.02, f"{avg:.5f}", ha="center", fontsize=7, color=GREEN)
    ax.text(i+width, p   + max(p_deltas)*0.02, f"{p:.5f}",   ha="center", fontsize=7, color="#ff9900")

# — Panel 2: National annual projection gauge ─────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("National Annual CO2e Saved [Avg]", fontsize=10, pad=8)
ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.02), 0.9, 0.94,
             boxstyle="round,pad=0.02", linewidth=2,
             edgecolor=GREEN, facecolor="#1a1a1a"))
ax.text(0.5, 0.80, f"{nat_annual_co2e_avg/1000:,.1f}",
        ha="center", va="center", fontsize=28, fontweight="bold", color=GREEN)
ax.text(0.5, 0.63, "tonnes CO2e / year",
        ha="center", va="center", fontsize=12, color="#cccccc")
ax.text(0.5, 0.48, f"Range: {nat_annual_co2e_Q/1000:,.1f} – {nat_annual_co2e_P/1000:,.1f} t CO2e/yr [Q–P]",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.34, f"Fuel saved: {nat_annual_litres_avg:,.0f} L/year",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.20, f"≈ {trees_avg:,.0f} trees absorbing CO2 for 1 year",
        ha="center", va="center", fontsize=9, color="#aaaaaa")
ax.text(0.5, 0.07, "★ Illustrative framing only — EPA GHG Equivalencies Calculator",
        ha="center", va="center", fontsize=7, color="#555555")

plt.tight_layout()
plt.savefig("national_environmental_primary.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → national_environmental_primary.png")

---
## Section 6 — P–Q Range + Sub-Models

In [ ]:
print("══ P–Q Range — National Annual CO2e Saved ══════════════════════")
print(f"  [Q] Conservative : {nat_annual_co2e_Q/1000:,.3f} tonnes CO2e/yr  |  {trees_Q:,.0f} trees")
print(f"  [Avg] Primary    : {nat_annual_co2e_avg/1000:,.3f} tonnes CO2e/yr  |  {trees_avg:,.0f} trees")
print(f"  [P] Aggressive   : {nat_annual_co2e_P/1000:,.3f} tonnes CO2e/yr  |  {trees_P:,.0f} trees")

In [ ]:
# ── Chart: P–Q range + trees ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("National Environmental Impact — P–Q Savings Range",
             fontsize=12, fontweight="bold", color="white", y=1.02)

bound_labels = ["Q\n(Conservative)", "Avg\n(Primary)", "P\n(Aggressive)"]
annual_co2e  = [nat_annual_co2e_Q/1000, nat_annual_co2e_avg/1000, nat_annual_co2e_P/1000]
trees_vals   = [trees_Q, trees_avg, trees_P]
bound_colors = ["#4488ff", GREEN, "#ff9900"]

# — Panel 1: Annual CO2e saved ────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(bound_labels, annual_co2e, color=bound_colors, width=0.4, zorder=3)
ax.set_title("Annual CO2e Saved (tonnes)", fontsize=11, pad=10)
ax.set_ylabel("tonnes CO2e")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.1f} t"))
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, annual_co2e):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(annual_co2e)*0.01,
            f"{val:,.2f} t", ha="center", va="bottom", fontsize=9, color="white")

# — Panel 2: Equivalent trees ─────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(bound_labels, trees_vals, color=bound_colors, width=0.4, zorder=3)
ax.set_title("Equivalent Trees (CO2e absorption/year)", fontsize=11, pad=10)
ax.set_ylabel("number of trees")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis="y", zorder=0)
for bar, val in zip(bars, trees_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(trees_vals)*0.01,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=9, color="white")
ax.text(0.5, -0.12, "★ Illustrative only — EPA GHG Equivalencies (McPherson et al. 2016)",
        ha="center", transform=ax.transAxes, fontsize=7, color="#555555")

plt.tight_layout()
plt.savefig("national_environmental_range.png", dpi=150, bbox_inches="tight",
            facecolor="#0f0f0f")
plt.show()
print("✅ Saved → national_environmental_range.png")

---
## Section 7 — Master Summary + Save JSON

In [ ]:
# ── Master summary table ──────────────────────────────────────────────────────

summary = pd.DataFrame({
    "Metric": [
        "Emission factor (kg CO2e/km)",
        "HD daily CO2e saved [Q]",
        "HD daily CO2e saved [Avg]",
        "HD daily CO2e saved [P]",
        "HD annual CO2e saved [Avg] (tonnes)",
        "HD daily fuel saved [Q–P] (litres)",
        "HYD daily CO2e saved [Q]",
        "HYD daily CO2e saved [Avg]",
        "HYD daily CO2e saved [P]",
        "HYD annual CO2e saved [Avg] (tonnes)",
        "HYD daily fuel saved [Q–P] (litres)",
        "National daily CO2e saved [Q]",
        "National daily CO2e saved [Avg]",
        "National daily CO2e saved [P]",
        "National annual CO2e saved [Q] (tonnes)",
        "National annual CO2e saved [Avg] (tonnes)",
        "National annual CO2e saved [P] (tonnes)",
        "National annual fuel saved [Avg] (litres)",
        "Equivalent trees [Q]",
        "Equivalent trees [Avg]",
        "Equivalent trees [P]",
    ],
    "Value": [
        f"{EMISSION_FACTOR_PER_KM} kg CO2e/km",
        f"{HD_DELTA_EMISSIONS_Q:.2f} kg CO2e/day",
        f"{HD_DELTA_EMISSIONS_AVG:.2f} kg CO2e/day",
        f"{HD_DELTA_EMISSIONS_P:.2f} kg CO2e/day",
        f"{HD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000:.3f} t CO2e/yr",
        f"{HD_DELTA_LITRES_Q:.2f} – {HD_DELTA_LITRES_P:.2f} L/day",
        f"{HYD_DELTA_EMISSIONS_Q:.2f} kg CO2e/day",
        f"{HYD_DELTA_EMISSIONS_AVG:.2f} kg CO2e/day",
        f"{HYD_DELTA_EMISSIONS_P:.2f} kg CO2e/day",
        f"{HYD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000:.3f} t CO2e/yr",
        f"{HYD_DELTA_LITRES_Q:.2f} – {HYD_DELTA_LITRES_P:.2f} L/day",
        f"{nat_daily_co2e_Q:.2f} kg CO2e/day",
        f"{nat_daily_co2e_avg:.2f} kg CO2e/day",
        f"{nat_daily_co2e_P:.2f} kg CO2e/day",
        f"{nat_annual_co2e_Q/1000:.3f} t CO2e/yr",
        f"{nat_annual_co2e_avg/1000:.3f} t CO2e/yr",
        f"{nat_annual_co2e_P/1000:.3f} t CO2e/yr",
        f"{nat_annual_litres_avg:,.0f} litres/yr",
        f"{trees_Q:,.0f} trees",
        f"{trees_avg:,.0f} trees",
        f"{trees_P:,.0f} trees",
    ]
})

print("══ Environmental Impact — Master Summary Table ═════════════════")
print(summary.to_string(index=False))

# ── Save JSON ─────────────────────────────────────────────────────────────────
env_metrics = {
    "emission_factor_kg_co2e_per_km": EMISSION_FACTOR_PER_KM,
    "tree_co2_kg_per_year": TREE_CO2_KG_PER_YEAR,
    "HD": {
        "daily_co2e_saved_kg": {"Q": round(HD_DELTA_EMISSIONS_Q,2), "Avg": round(HD_DELTA_EMISSIONS_AVG,2), "P": round(HD_DELTA_EMISSIONS_P,2)},
        "annual_co2e_saved_tonnes": {"Q": round(HD_DELTA_EMISSIONS_Q*DAYS_PER_YEAR/1000,3), "Avg": round(HD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000,3), "P": round(HD_DELTA_EMISSIONS_P*DAYS_PER_YEAR/1000,3)},
        "daily_fuel_saved_litres": {"Q": round(HD_DELTA_LITRES_Q,2), "P": round(HD_DELTA_LITRES_P,2)}
    },
    "HYD": {
        "daily_co2e_saved_kg": {"Q": round(HYD_DELTA_EMISSIONS_Q,2), "Avg": round(HYD_DELTA_EMISSIONS_AVG,2), "P": round(HYD_DELTA_EMISSIONS_P,2)},
        "annual_co2e_saved_tonnes": {"Q": round(HYD_DELTA_EMISSIONS_Q*DAYS_PER_YEAR/1000,3), "Avg": round(HYD_DELTA_EMISSIONS_AVG*DAYS_PER_YEAR/1000,3), "P": round(HYD_DELTA_EMISSIONS_P*DAYS_PER_YEAR/1000,3)},
        "daily_fuel_saved_litres": {"Q": round(HYD_DELTA_LITRES_Q,2), "P": round(HYD_DELTA_LITRES_P,2)}
    },
    "National": {
        "daily_co2e_saved_kg": {"Q": round(nat_daily_co2e_Q,2), "Avg": round(nat_daily_co2e_avg,2), "P": round(nat_daily_co2e_P,2)},
        "annual_co2e_saved_tonnes": {"Q": round(nat_annual_co2e_Q/1000,3), "Avg": round(nat_annual_co2e_avg/1000,3), "P": round(nat_annual_co2e_P/1000,3)},
        "annual_fuel_saved_litres": {"Avg": round(nat_annual_litres_avg,0)},
        "equivalent_trees": {"Q": round(trees_Q,0), "Avg": round(trees_avg,0), "P": round(trees_P,0)}
    }
}

with open("environmental_metrics.json", "w") as f:
    json.dump(env_metrics, f, indent=2)
print("\n✅ Saved → environmental_metrics.json")